## Fine‑tune Modelo de clasificación basdado en transformers: 
### distilbert-base-uncased (DistilBERT)

In [1]:
using_colab = True

In [2]:
# Check if running in Colab and setup paths
import os
if using_colab:
    import gdown
    url = "https://drive.google.com/uc?id=1T1gBrg3aohHl2XV0deNPXhD7OzNkrcJT"
    DATASET_PATH = "MTS-Dialog-TrainingSet.csv"
    gdown.download(url, DATASET_PATH, quiet=False)
    
else:
    using_colab = False
    DATASET_PATH = "../../../dataset/MTS-Dialog-TrainingSet.csv"

print(f"Running in Colab: {using_colab}")
print(f"Dataset path: {DATASET_PATH}")

Downloading...
From: https://drive.google.com/uc?id=1T1gBrg3aohHl2XV0deNPXhD7OzNkrcJT
To: /content/MTS-Dialog-TrainingSet.csv
100%|██████████| 1.04M/1.04M [00:00<00:00, 137MB/s]

Running in Colab: True
Dataset path: MTS-Dialog-TrainingSet.csv


In [3]:
import pandas as pd

In [4]:
import re
import unicodedata
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import DistilBertTokenizer, DistilBertModel


In [5]:
# carga del dataset
df = pd.read_csv(DATASET_PATH)

In [6]:
# preprocesamiento para BERT
def normalize_for_bert(s):
    if pd.isna(s):
        return ""
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r'\b(Doctor|Doctor_2|Patient|Guest_family(_\d)?|Guest_clinician)[:\-]\s*', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

df['text_for_bert'] = df['dialogue'].apply(normalize_for_bert)


X = df['text_for_bert']
y = df['section_header']

# Encode 
le = LabelEncoder()
y_encoded = le.fit_transform(y)


X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42
)

In [7]:
# carga del tokenizer y modelo
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_name = "distilbert-base-uncased"

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(le.classes_),
    problem_type="single_label_classification"
)

# Tokenización
def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        padding='max_length',
        truncation=True,
        max_length=512
    )

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [8]:
# datasets
train_dataset = Dataset.from_dict({'text': X_train.tolist(), 'label': y_train.tolist()})
test_dataset = Dataset.from_dict({'text': X_test.tolist(), 'label': y_test.tolist()})

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])

Map:   0%|          | 0/960 [00:00<?, ? examples/s]

Map:   0%|          | 0/241 [00:00<?, ? examples/s]

In [9]:
# métricas de evaluación
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1_macro': f1_score(labels, predictions, average='macro')
    }

In [10]:
# configuración del entrenamiento
training_args = TrainingArguments(
    output_dir='./results_clinicalbert',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_dir='./logs',
    logging_steps=10,
    seed=42,
    fp16=True,
    gradient_accumulation_steps=2,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
if using_colab:   
    import os
    import wandb

    # in order to print results using colab API key for wandb is needed as local file usage as .env files or datasets are not well built for the moment
    # if someone else is running the notebook, dont commit your key,make sure to remove it before pushing to the repo repo 
    os.environ["WANDB_API_KEY"] = ""

    wandb.login()

In [12]:
# Fine-tune
trainer.train()

# Evaluación
results = trainer.evaluate()
print(results)



Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,1.859100,1.634654,0.609959,0.184997
2,1.381700,1.288926,0.697095,0.278254
3,1.202500,1.188819,0.746888,0.315671


{'eval_loss': 1.1888186931610107, 'eval_accuracy': 0.7468879668049793, 'eval_f1_macro': 0.3156711430663626, 'eval_runtime': 0.9101, 'eval_samples_per_second': 264.808, 'eval_steps_per_second': 34.062, 'epoch': 3.0}


In [13]:
if using_colab:
    # As mentioned before, Colab requires specific handling local files management, if needed, you can mount
    # on personal drive and save it there.
    print("Using colab virtual hardware - model persistency not implemented")

else:
    print("Running locally, model persistency implemented")
    # Guardar modelo
    model.save_pretrained('./finetuned_distilbert')
    tokenizer.save_pretrained('./finetuned_distilbert')

    # Guardar encoder de labels
    import pickle
    with open('./finetuned_distilbert/label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)

Using colab virtual hardware - model persistency not implemented


CONCLUSIONES

MODEL INFERENCE TEST

validation set import

In [ ]:
# Check if running in Colab and setup paths
import os
if using_colab:
    import gdown
    url = "https://drive.google.com/file/d/1L-Fz5jlc9g1G43jwg-l6Ib23D23Mmxgj/view?usp=sharing"
    DATASET_PATH_2 = "MTS-Dialog-ValidationSet.csv"
    gdown.download(url, DATASET_PATH_2, quiet=False)
    
else:
    using_colab = False
    DATASET_PATH_2 = "../../../dataset/MTS-Dialog-ValidationSet.csv"

print(f"Running in Colab: {using_colab}")
print(f"Dataset path: {DATASET_PATH}")

In [ ]:
import pandas as pd
from transformers import pipeline
import re, unicodedata

def normalize_for_bert(s):
    s = unicodedata.normalize("NFKC", str(s))
    s = re.sub(r'\b(Doctor|Doctor_2|Patient|Guest_family(_\d)?|Guest_clinician)[:\-]\s*', '', s, flags=re.I)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

# Load label encoder and model

clf = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0  # use -1 for CPU
)

# Load a validation sample
df_val = pd.read_csv(DATASET_PATH_2)
text = normalize_for_bert(df_val.loc[0, "dialogue"])  # any row from validation set

# Predict
out = clf(text)[0]
pred_label = le.inverse_transform([int(out['label'].split('_')[-1])])[0]
print(f"Predicted: {pred_label} | Confidence: {out['score']:.3f}")

Device set to use cuda:0


Predicted: GENHX | Confidence: 0.497
